# SemaforoIA | Entrenamiento y evaluación

Ajustamos YOLOv8n, preentrenado para detección de objetos, con imágenes etiquetadas de tráfico. Una red MLP aprende duraciones de luz verde a partir de experimentos de SUMO. Streamlit conecta ambas redes para mostrar el conteo, los tiempos predichos y las fases de un cruce virtual.

### 1. Dependencias y hardware

Seleccionamos la GPU T4 en Colab cuando hay cupo. PyTorch y su CUDA se conservan del entorno de Colab para mantener la compatibilidad con la GPU.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics==8.4.215', 'scikit-learn>=1.6,<2',
])

In [ ]:
import os
import platform
from pathlib import Path
import torch
import ultralytics

cpu_info = Path('/proc/cpuinfo')
cpu = platform.processor() or platform.machine()
if cpu_info.is_file():
    cpu = next((line.split(':', 1)[1].strip() for line in cpu_info.read_text().splitlines()
                if line.startswith('model name')), cpu)

print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__} | Ultralytics: {ultralytics.__version__}')
print(f'CUDA de PyTorch: {torch.version.cuda} | CUDA disponible: {torch.cuda.is_available()}')
print(f'CPU: {cpu} | Hilos lógicos: {os.cpu_count()}')
for gpu_index in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(gpu_index)
    print(f'GPU {gpu_index}: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB')
if not torch.cuda.is_available():
    print('GPU: sin asignar. Puedes preparar los datos; el bloque de entrenamiento requiere CUDA.')

### 2. Google Drive, semilla y parámetros

Colab suele guardar este notebook automáticamente en Drive al abrirlo como notebook propio. El código guarda datos y resultados en `Mi unidad/SemaforoIA`. Si se conserva allí también el notebook, todos los archivos quedan juntos. La semilla fija las particiones y los generadores aleatorios. Las versiones del entorno y el hardware también influyen en la reproducibilidad.

In [ ]:
import hashlib
import json
import random
import shutil
from collections import Counter

import numpy as np
import pandas as pd
from google.colab import drive, userdata
from IPython.display import display

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/SemaforoIA')
WORK = Path('/content/semaforoia')
for folder in (ROOT, WORK, ROOT / 'modelos', ROOT / 'auditoria'):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
EPOCHS = 30
CV_EPOCHS = 10  # Cada pliegue tiene el mismo presupuesto inicial de 10 épocas.
FOLDS = 3
IMGSZ = 640
BATCH = 0.60  # AutoBatch utiliza aproximadamente el 60 % de la VRAM de la GPU.
RUN_ID = 'v1_semilla42'
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def copy_atomic(source, target):
    """Copia completa antes de reemplazar el archivo persistente."""
    target.parent.mkdir(parents=True, exist_ok=True)
    pending = target.with_name(target.name + '.part')
    shutil.copy2(source, pending)
    pending.replace(target)

def write_json(target, value):
    pending = target.with_name(target.name + '.part')
    pending.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    pending.replace(target)

print(f'Drive: {ROOT} | Semilla: {SEED} | Épocas: {EPOCHS} | CV: {FOLDS} × {CV_EPOCHS}')

### 3. SUMO y datos del controlador neuronal

Simulamos dos vías con un carril por acceso y movimientos rectos. Generamos 192 escenarios de llegada y mezcla de vehículos, con 36 pares de verde por escenario. Cada par se prueba con las mismas llegadas durante 150 s, tras un calentamiento de 45 s. La etiqueta es el par que minimiza los segundos-vehículo de cola más 20 unidades por vehículo sin completar su viaje al cierre. El caché en Drive permite recuperar escenarios terminados.

El MLP recibe los vehículos visibles por vía, los pesados por vía, el total de ambas vías y la participación de la vía A. Se usan entradas disponibles también con fotografías. [Documentación de SUMO y libsumo](https://sumo.dlr.de/docs/Libsumo.html).

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'eclipse-sumo==1.28.0', 'libsumo==1.28.0'])
print(subprocess.check_output(['netconvert', '--version'], text=True).splitlines()[0])

# El notebook incluye una copia de los módulos del proyecto y funciona por sí solo.
MLP_CODE = WORK / 'codigo_controlador'
MLP_CODE.mkdir(exist_ok=True)
(MLP_CODE / 'traffic.py').write_text('"""Plan de fases para dos vías en la simulación académica."""\n\nfrom dataclasses import dataclass\nfrom math import isfinite\n\n\nMIN_GREEN, MAX_GREEN = 10, 35\nYELLOW, ALL_RED = 3, 2\n\n\n@dataclass(frozen=True)\nclass Phase:\n    a: str\n    b: str\n    seconds: int\n\n\ndef demonstration_greens(count_a: int, count_b: int) -> tuple[int, int]:\n    """Línea base proporcional, disponible para evaluación y demostración sin pesos."""\n    if min(count_a, count_b) < 0:\n        raise ValueError(\'El conteo de vehículos debe ser positivo o cero.\')\n    total = count_a + count_b\n    if total == 0:\n        return 0, 0\n    green_a = bounded_green(45 * count_a / total)\n    return green_a, 45 - green_a\n\n\ndef bounded_green(seconds: float) -> int:\n    """Convierte una predicción finita a segundos enteros dentro de los límites."""\n    if not isfinite(seconds):\n        raise ValueError(\'El controlador produjo una duración inválida.\')\n    return min(MAX_GREEN, max(MIN_GREEN, round(seconds)))\n\n\ndef build_cycle(greens: tuple[int, int]) -> tuple[Phase, ...]:\n    """Intercala fases seguras. La red solamente aporta los tiempos de verde."""\n    if greens == (0, 0):\n        return (Phase(\'Rojo\', \'Rojo\', 0),)\n    green_a, green_b = (bounded_green(seconds) for seconds in greens)\n    clearance = Phase(\'Rojo\', \'Rojo\', ALL_RED)\n    return (Phase(\'Verde\', \'Rojo\', green_a), Phase(\'Amarillo\', \'Rojo\', YELLOW), clearance,\n            Phase(\'Rojo\', \'Verde\', green_b), Phase(\'Rojo\', \'Amarillo\', YELLOW), clearance)\n\n\ndef phase_at(cycle: tuple[Phase, ...], elapsed: int) -> tuple[Phase, int]:\n    """Devuelve la fase activa y cuántos segundos faltan para el cambio."""\n    duration = sum(phase.seconds for phase in cycle)\n    if duration == 0:\n        return cycle[0], 0\n\n    position = max(0, elapsed) % duration\n    for phase in cycle:\n        if position < phase.seconds:\n            return phase, phase.seconds - position\n        position -= phase.seconds\n    raise RuntimeError(\'El plan de fases no se pudo recorrer.\')\n', encoding='utf-8')
(MLP_CODE / 'controller.py').write_text('"""Modelo de regresión para los tiempos de luz verde de dos vías."""\n\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\n\nfrom traffic import MAX_GREEN, MIN_GREEN, bounded_green\n\n\nFEATURES = (\'vehiculos_a\', \'vehiculos_b\', \'pesados_a\', \'pesados_b\', \'demanda_total\', \'proporcion_a\')\n\n\ndef traffic_features(count_a: int, count_b: int, heavy_a: int, heavy_b: int) -> tuple[float, ...]:\n    """Prepara variables que están disponibles tanto en SUMO como en una foto por vía."""\n    if not (0 <= heavy_a <= count_a and 0 <= heavy_b <= count_b):\n        raise ValueError(\'El conteo de vehículos pesados debe estar dentro del conteo total de cada vía.\')\n    total = count_a + count_b\n    return count_a, count_b, heavy_a, heavy_b, total, count_a / total if total else 0.5\n\n\nclass GreenTimeMLP(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.layers = nn.Sequential(\n            nn.Linear(len(FEATURES), 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2),\n        )\n\n    def forward(self, x):\n        return MIN_GREEN + (MAX_GREEN - MIN_GREEN) * torch.sigmoid(self.layers(x))\n\n\nclass TimeController:\n    def __init__(self, weights: Path):\n        saved = torch.load(weights, map_location=\'cpu\', weights_only=True)\n        if (saved[\'format_version\'] != 1 or tuple(saved[\'features\']) != FEATURES\n                or tuple(saved[\'green_limits\']) != (MIN_GREEN, MAX_GREEN)):\n            raise ValueError(\'El checkpoint MLP tiene una configuración incompatible.\')\n        self.model = GreenTimeMLP()\n        self.model.load_state_dict(saved[\'state_dict\'])\n        self.model.eval()\n        self.mean, self.scale = saved[\'mean\'], saved[\'scale\']\n        self.count_limits = saved[\'count_limits\']\n\n    def predict(self, count_a: int, count_b: int, heavy_a: int, heavy_b: int) -> tuple[int, int]:\n        features = traffic_features(count_a, count_b, heavy_a, heavy_b)\n        if count_a + count_b == 0:\n            return 0, 0\n        with torch.inference_mode():\n            normalized = (torch.tensor(features, dtype=torch.float32) - self.mean) / self.scale\n            green_a, green_b = self.model(normalized).tolist()\n        return bounded_green(green_a), bounded_green(green_b)\n', encoding='utf-8')
(MLP_CODE / 'sumo_experiment.py').write_text('"""Escenarios de un cruce de dos vías y evaluación de planes en Eclipse SUMO."""\n\nfrom dataclasses import dataclass\nfrom itertools import product\nfrom pathlib import Path\nimport random\nimport subprocess\nfrom xml.etree import ElementTree as ET\n\nfrom traffic import ALL_RED, build_cycle\n\n\nWARMUP = 45\nHORIZON = 150\nBACKLOG_WEIGHT = 20\nGREEN_GRID = tuple(product(range(10, 36, 5), repeat=2))\nINCOMING = (\'a_in\', \'b_in\')\nTLS = \'cruce\'\nCOLORS = {\'Verde\': \'G\', \'Amarillo\': \'y\', \'Rojo\': \'r\'}\n\n\ndef write_xml(path, root):\n    ET.ElementTree(root).write(path, encoding=\'utf-8\', xml_declaration=True)\n\n\ndef make_network(folder: Path) -> Path:\n    """Construye dos movimientos rectos y un único cruce semaforizado."""\n    folder.mkdir(parents=True, exist_ok=True)\n    nodes = ET.Element(\'nodes\')\n    for name, x, y in [(\'oeste\', -280, 0), (\'este\', 65, 0), (\'sur\', 0, -280),\n                       (\'norte\', 0, 65), (TLS, 0, 0)]:\n        attrs = {\'type\': \'traffic_light\'} if name == TLS else {}\n        ET.SubElement(nodes, \'node\', id=name, x=str(x), y=str(y), **attrs)\n    edges = ET.Element(\'edges\')\n    for name, origin, destination in [(\'a_in\', \'oeste\', TLS), (\'a_out\', TLS, \'este\'),\n                                       (\'b_in\', \'sur\', TLS), (\'b_out\', TLS, \'norte\')]:\n        ET.SubElement(edges, \'edge\', {\'id\': name, \'from\': origin, \'to\': destination,\n                                     \'numLanes\': \'1\', \'speed\': \'13.89\'})\n    links = ET.Element(\'connections\')\n    for approach in (\'a\', \'b\'):\n        ET.SubElement(links, \'connection\', {\'from\': f\'{approach}_in\', \'to\': f\'{approach}_out\',\n                                            \'fromLane\': \'0\', \'toLane\': \'0\'})\n    for name, document in [(\'nodos\', nodes), (\'vias\', edges), (\'giros\', links)]:\n        write_xml(folder / f\'{name}.xml\', document)\n    net = folder / \'cruce.net.xml\'\n    subprocess.run([\'netconvert\', \'--node-files\', str(folder / \'nodos.xml\'),\n                    \'--edge-files\', str(folder / \'vias.xml\'), \'--connection-files\', str(folder / \'giros.xml\'),\n                    \'--output-file\', str(net)], check=True, capture_output=True, text=True)\n    return net\n\n\ndef make_routes(folder: Path, scenario_seed: int) -> tuple[Path, int]:\n    """Genera llegadas y mezcla de vehículos. Cada plan comparte las mismas llegadas."""\n    rng = random.Random(scenario_seed)\n    routes = ET.Element(\'routes\')\n    ET.SubElement(routes, \'vType\', id=\'liviano\', vClass=\'passenger\', sigma=\'0.3\')\n    ET.SubElement(routes, \'vType\', id=\'pesado\', vClass=\'truck\', length=\'9\', accel=\'1.1\', sigma=\'0.3\')\n    departures = []\n    for approach in (\'a\', \'b\'):\n        ET.SubElement(routes, \'route\', id=approach, edges=f\'{approach}_in {approach}_out\')\n        demand = rng.uniform(0.025, 0.52)\n        heavy_share = rng.uniform(0.03, 0.40)\n        # El tramo posterior añade variación al flujo observable durante el calentamiento.\n        future = min(0.60, demand * rng.uniform(0.65, 1.35))\n        arrivals = [(0, WARMUP, demand), (WARMUP, WARMUP + HORIZON - 15, future)]\n        for start, stop, probability in arrivals:\n            for second in range(start, stop):\n                if rng.random() < probability:\n                    vehicle = \'pesado\' if rng.random() < heavy_share else \'liviano\'\n                    departures.append((second, approach, vehicle))\n    for index, (second, approach, vehicle) in enumerate(sorted(departures)):\n        ET.SubElement(routes, \'vehicle\', id=f\'v{index}\', type=vehicle, route=approach,\n                      depart=str(second), departSpeed=\'max\')\n    path = folder / \'escenario.rou.xml\'\n    write_xml(path, routes)\n    return path, len(departures)\n\n\n@dataclass(frozen=True)\nclass SimulationResult:\n    counts: tuple[int, int, int, int]\n    queue_seconds: int\n    arrived: int\n    unfinished: int\n    collision_events: int\n\n    @property\n    def objective(self):\n        return self.queue_seconds + BACKLOG_WEIGHT * self.unfinished\n\n\nclass IntersectionSimulation:\n    def __init__(self, net: Path):\n        import libsumo\n\n        self.sumo = libsumo\n        self.net = net\n\n    def run(self, routes: Path, scheduled: int, scenario_seed: int, greens: tuple[int, int]):\n        sim = self.sumo\n        sim.start([\'sumo\', \'-n\', str(self.net), \'-r\', str(routes), \'--seed\', str(scenario_seed),\n                   \'--no-step-log\', \'--no-warnings\', \'--duration-log.disable\', \'--time-to-teleport\', \'-1\'])\n        try:\n            controlled = sim.trafficlight.getControlledLinks(TLS)\n            order = [INCOMING.index(links[0][0].rsplit(\'_\', 1)[0]) for links in controlled]\n            if sorted(order) != [0, 1]:\n                raise RuntimeError(\'SUMO debe exponer dos señales controladas, una por vía.\')\n            sim.trafficlight.setRedYellowGreenState(TLS, \'rr\')\n            for _ in range(WARMUP):\n                sim.simulationStep()\n            ids_a, ids_b = [sim.edge.getLastStepVehicleIDs(edge) for edge in INCOMING]\n            counts = (len(ids_a), len(ids_b), sum(sim.vehicle.getTypeID(v) == \'pesado\' for v in ids_a),\n                      sum(sim.vehicle.getTypeID(v) == \'pesado\' for v in ids_b))\n            states = []\n            for phase in build_cycle(greens):\n                signal = (COLORS[phase.a], COLORS[phase.b])\n                states.extend([\'\'.join(signal[index] for index in order)] * phase.seconds)\n            states = states or [\'rr\']\n            # La primera autorización de paso también tiene un intervalo todo rojo.\n            previous = \'rr\'\n            queue = arrived = collisions = 0\n            for second in range(HORIZON):\n                signal = \'rr\' if second < ALL_RED else states[(second - ALL_RED) % len(states)]\n                if signal != previous:\n                    sim.trafficlight.setRedYellowGreenState(TLS, signal)\n                    previous = signal\n                sim.simulationStep()\n                queue += sum(sim.edge.getLastStepHaltingNumber(edge) for edge in INCOMING)\n                arrived += sim.simulation.getArrivedNumber()\n                collisions += sim.simulation.getCollidingVehiclesNumber()\n            return SimulationResult(counts, queue, arrived, scheduled - arrived, collisions)\n        finally:\n            sim.close()\n', encoding='utf-8')
(MLP_CODE / 'train_controller.py').write_text('"""Generación, ajuste y evaluación reproducible del controlador neuronal en Colab."""\n\nfrom copy import deepcopy\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport json\nimport random\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\n\nfrom controller import FEATURES, GreenTimeMLP, TimeController, traffic_features\nfrom sumo_experiment import BACKLOG_WEIGHT, GREEN_GRID, HORIZON, WARMUP, IntersectionSimulation, make_network, make_routes\nfrom traffic import ALL_RED, MAX_GREEN, MIN_GREEN, YELLOW, demonstration_greens\n\n\ndef write_report(path, content):\n    partial = path.with_suffix(path.suffix + \'.part\')\n    partial.write_text(json.dumps(content, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    partial.replace(path)\n\n\ndef generate_sumo_data(work: Path, output: Path, scenarios=192, seed=42):\n    """Simula cada combinación y recupera escenarios ya calculados tras una desconexión."""\n    import libsumo\n\n    output.mkdir(parents=True, exist_ok=True)\n    cache = output / \'escenarios\'\n    cache.mkdir(exist_ok=True)\n    simulation = IntersectionSimulation(make_network(work))\n    spec = {\'semilla\': seed, \'escenarios\': scenarios, \'calentamiento_s\': WARMUP,\n            \'horizonte_s\': HORIZON, \'penalizacion_pendientes\': BACKLOG_WEIGHT, \'version_sumo\': libsumo.getVersion()[1],\n            \'rejilla\': [list(pair) for pair in GREEN_GRID], \'amarillo_s\': YELLOW, \'todo_rojo_s\': ALL_RED}\n    manifest = output / \'protocolo.json\'\n    if manifest.exists() and json.loads(manifest.read_text(encoding=\'utf-8\')) != spec:\n        raise ValueError(\'Para cambiar el protocolo, usa otra carpeta de resultados.\')\n    write_report(manifest, spec)\n    labels, trials = [], []\n    for index in range(scenarios):\n        scenario = seed * 100_000 + index\n        cached = cache / f\'{scenario}.json\'\n        if cached.exists():\n            record = json.loads(cached.read_text(encoding=\'utf-8\'))\n        else:\n            routes, scheduled = make_routes(work, scenario)\n            plans = []\n            for a, b in GREEN_GRID:\n                result = simulation.run(routes, scheduled, scenario, (a, b))\n                if result.collision_events:\n                    raise RuntimeError(f\'SUMO registró una colisión en el escenario {scenario}.\')\n                plans.append({\'escenario\': scenario, \'verde_a\': a, \'verde_b\': b,\n                              **asdict(result), \'objetivo\': result.objective})\n            if len({plan[\'counts\'] for plan in plans}) != 1:\n                raise RuntimeError(\'Las pruebas de un escenario no empezaron con el mismo estado.\')\n            # El desempate prefiere menor espera y, después, un ciclo más corto.\n            best = min(plans, key=lambda p: (p[\'objetivo\'], p[\'queue_seconds\'],\n                                             p[\'verde_a\'] + p[\'verde_b\'], p[\'verde_a\']))\n            features = traffic_features(*best[\'counts\'])\n            label = {\'escenario\': scenario, **dict(zip(FEATURES, features)), \'verde_a\': best[\'verde_a\'],\n                     \'verde_b\': best[\'verde_b\'], \'objetivo_rejilla\': best[\'objetivo\']}\n            record = {\'etiqueta\': label, \'planes\': plans}\n            write_report(cached, record)\n        labels.append(record[\'etiqueta\'])\n        trials.extend(record[\'planes\'])\n        if (index + 1) % 12 == 0 or index + 1 == scenarios:\n            print(f\'SUMO · {index + 1}/{scenarios} escenarios\', flush=True)\n    dataset = pd.DataFrame(labels)\n    dataset.to_csv(output / \'datos_mlp.csv\', index=False)\n    pd.DataFrame(trials).to_csv(output / \'candidatos_sumo.csv\', index=False)\n    return dataset\n\n\ndef train_mlp(dataset: pd.DataFrame, output: Path, weights: Path, seed=42, epochs=120):\n    """Entrena, selecciona por pérdida de validación y mide sobre escenarios de prueba."""\n    from sklearn.model_selection import train_test_split\n    import matplotlib.pyplot as plt\n\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.set_num_threads(2)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    # El estado sin vehículos activa el todo rojo y queda fuera del ajuste y evaluación MLP.\n    eligible = dataset[dataset[\'demanda_total\'] > 0].copy()\n    train, rest = train_test_split(eligible, test_size=0.30, random_state=seed)\n    valid, test = train_test_split(rest, test_size=0.50, random_state=seed)\n    membership = pd.concat([table[[\'escenario\']].assign(particion=name)\n                            for name, table in [(\'train\', train), (\'valid\', valid), (\'test\', test)]])\n    membership.to_csv(output / \'particiones.csv\', index=False)\n\n    def tensors(table):\n        return (torch.tensor(table[list(FEATURES)].to_numpy(), dtype=torch.float32),\n                torch.tensor(table[[\'verde_a\', \'verde_b\']].to_numpy(), dtype=torch.float32))\n\n    x_train, y_train = tensors(train)\n    x_valid, y_valid = tensors(valid)\n    mean, scale = x_train.mean(dim=0), x_train.std(dim=0, unbiased=False).clamp_min(1e-6)\n    train_batches = DataLoader(TensorDataset((x_train - mean) / scale, y_train), batch_size=32, shuffle=True,\n                               generator=torch.Generator().manual_seed(seed))\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = GreenTimeMLP().to(device)\n    loss_fn = nn.SmoothL1Loss()\n    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.001)\n    x_valid, y_valid = ((x_valid - mean) / scale).to(device), y_valid.to(device)\n    history, best_loss = [], float(\'inf\')\n    best_state = None\n    for epoch in range(1, epochs + 1):\n        model.train()\n        total_loss = 0.0\n        for x, y in train_batches:\n            optimizer.zero_grad()\n            loss = loss_fn(model(x.to(device)), y.to(device))\n            loss.backward()\n            optimizer.step()\n            total_loss += loss.item() * len(x)\n        model.eval()\n        with torch.inference_mode():\n            val_loss = loss_fn(model(x_valid), y_valid).item()\n        history.append({\'epoca\': epoch, \'entrenamiento\': total_loss / len(train), \'validacion\': val_loss})\n        if val_loss < best_loss:\n            best_loss, best_epoch = val_loss, epoch\n            best_state = deepcopy(model.state_dict())\n    if best_state is None:\n        raise RuntimeError(\'El MLP no produjo un checkpoint con pérdida finita.\')\n\n    weights.parent.mkdir(parents=True, exist_ok=True)\n    pending = weights.with_suffix(\'.pt.part\')\n    torch.save({\'format_version\': 1, \'features\': list(FEATURES), \'green_limits\': [MIN_GREEN, MAX_GREEN],\n                \'state_dict\': {key: value.cpu() for key, value in best_state.items()}, \'mean\': mean, \'scale\': scale,\n                \'count_limits\': [int(train[\'vehiculos_a\'].max()), int(train[\'vehiculos_b\'].max())],\n                \'best_epoch\': best_epoch, \'seed\': seed}, pending)\n    pending.replace(weights)\n    history = pd.DataFrame(history)\n    history.to_csv(output / \'aprendizaje.csv\', index=False)\n    ax = history.plot(x=\'epoca\', y=[\'entrenamiento\', \'validacion\'], ylabel=\'Pérdida Smooth L1\', xlabel=\'Época\')\n    ax.figure.savefig(output / \'curva_mlp.png\', dpi=140, bbox_inches=\'tight\')\n    plt.close(ax.figure)\n\n    predictor = TimeController(weights)\n    predictions = [predictor.predict(row.vehiculos_a, row.vehiculos_b, row.pesados_a, row.pesados_b)\n                   for row in test.itertuples()]\n    test = test.assign(predicho_a=[row[0] for row in predictions], predicho_b=[row[1] for row in predictions])\n    test.to_csv(output / \'predicciones_prueba.csv\', index=False)\n    error = np.array(predictions) - test[[\'verde_a\', \'verde_b\']].to_numpy()\n    metrics = {\'entrenamiento\': len(train), \'validacion\': len(valid), \'prueba\': len(test),\n               \'epocas\': epochs, \'mejor_epoca\': best_epoch, \'dispositivo\': str(device),\n               \'mae_a_s\': float(np.abs(error[:, 0]).mean()), \'mae_b_s\': float(np.abs(error[:, 1]).mean()),\n               \'rmse_a_s\': float(np.sqrt((error[:, 0] ** 2).mean())),\n               \'rmse_b_s\': float(np.sqrt((error[:, 1] ** 2).mean()))}\n    write_report(output / \'metricas_mlp.json\', metrics)\n    print(\'Métricas del MLP:\', json.dumps(metrics, ensure_ascii=False), flush=True)\n    return test\n\n\ndef evaluate_sumo(test: pd.DataFrame, work: Path, output: Path):\n    """Re-simula cada predicción y sus referencias con las llegadas de la prueba reservada."""\n    sim = IntersectionSimulation(make_network(work))\n    comparisons = []\n    for row in test.itertuples():\n        routes, scheduled = make_routes(work, row.escenario)\n        plans = {\'mlp\': (row.predicho_a, row.predicho_b), \'fijo\': (22, 23),\n                 \'proporcional\': demonstration_greens(row.vehiculos_a, row.vehiculos_b),\n                 \'mejor_rejilla\': (row.verde_a, row.verde_b)}\n        for method, greens in plans.items():\n            result = sim.run(routes, scheduled, row.escenario, greens)\n            comparisons.append({\'escenario\': row.escenario, \'metodo\': method, \'verde_a\': greens[0],\n                                \'verde_b\': greens[1], **asdict(result), \'objetivo\': result.objective})\n    comparisons = pd.DataFrame(comparisons)\n    comparisons.to_csv(output / \'comparacion_prueba_sumo.csv\', index=False)\n    summary = comparisons.groupby(\'metodo\')[[\'objetivo\', \'queue_seconds\', \'arrived\', \'unfinished\', \'collision_events\']].mean()\n    summary.to_csv(output / \'resumen_prueba_sumo.csv\')\n    print(summary.to_string(), flush=True)\n    return summary\n', encoding='utf-8')
sys.path.insert(0, str(MLP_CODE))

from train_controller import generate_sumo_data, train_mlp, evaluate_sumo

MLP_OUTPUT = ROOT / 'resultados' / 'controlador_v1'
SUMO_WORK = WORK / 'sumo'
MLP_WEIGHTS = ROOT / 'modelos' / 'controlador_tiempos.pt'
sumo_data = generate_sumo_data(SUMO_WORK, MLP_OUTPUT, scenarios=192, seed=SEED)
display(sumo_data.head())

### 4. Entrenamiento de la MLP con PyTorch

Separamos los escenarios en 70 % de entrenamiento, 15 % de validación y 15 % de prueba. La normalización se estima con entrenamiento. La red tiene dos capas ocultas de 64 y 32 neuronas y dos salidas, una por cada vía. Usamos AdamW y Smooth L1 durante 120 épocas. Conservamos la época con menor pérdida de validación.

In [ ]:
mlp_test = train_mlp(sumo_data, MLP_OUTPUT, MLP_WEIGHTS, seed=SEED, epochs=120)
from IPython.display import Image
display(Image(filename=str(MLP_OUTPUT / 'curva_mlp.png'), width=750))
print('Checkpoint MLP en Drive:', MLP_WEIGHTS)

### 5. Prueba del MLP y comparación en SUMO

Medimos MAE y RMSE en segundos para las dos salidas. Volvemos a simular los planes del MLP sobre escenarios reservados y los comparamos con un semáforo fijo, un reparto proporcional y el mejor par de la rejilla. Se reporta costo, cola acumulada, vehículos que completaron el viaje, pendientes y colisiones registradas.

In [ ]:
display(evaluate_sumo(mlp_test, SUMO_WORK, MLP_OUTPUT))

### 6. Descargar el controlador neuronal

In [ ]:
from google.colab import files
from ipywidgets import Button

def download_button(path, label):
    button = Button(description=label, disabled=not path.is_file(), layout={'width': 'max-content'})
    button.on_click(lambda _: files.download(str(path)))
    display(button)

download_button(MLP_WEIGHTS, 'Descargar controlador_tiempos.pt')

### 7. Dataset YOLOv8 de Roboflow

Usamos **Intersection-traffic-detection v1**, de Remas Workspace, con licencia CC BY 4.0. Roboflow publica 5.429 imágenes de entrenamiento, 603 de validación y 671 de prueba. Se conserva el contenido exportado y sus tres particiones. El modelo lee los archivos desde el disco rápido de Colab.

La celda usa el ZIP `dataset_semaforoia_yolov8.zip` si ya existe en la carpeta `SemaforoIA` de Drive. También puede descargar la versión exacta si el secreto `ROBOFLOW_API_KEY` existe y tiene habilitado el acceso para este notebook. [Dataset y licencia](https://universe.roboflow.com/remas-workspace/intersection-traffic-detection-sf6fy/dataset/1).

In [ ]:
from zipfile import ZipFile
import yaml

EXPECTED = {'train': 5429, 'valid': 603, 'test': 671}
IMAGE_TYPES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
ZIP = ROOT / 'dataset_semaforoia_yolov8.zip'
DATA = WORK / 'dataset'
DRIVE_DATA = ROOT / 'dataset'

def dataset_counts(folder):
    return {split: sum(p.suffix.lower() in IMAGE_TYPES for p in
                       (folder / split / 'images').glob('*') if p.is_file())
            for split in EXPECTED}

def check_export(folder):
    counts = dataset_counts(folder)
    if counts != EXPECTED or not (folder / 'data.yaml').is_file():
        raise ValueError(f'La exportación YOLOv8 v1 debe tener {EXPECTED}. Se encontró {counts}.')
    for split in EXPECTED:
        if not (folder / split / 'labels').is_dir():
            raise ValueError(f'No se encontró la carpeta de etiquetas de {split}.')
    return counts

if not (DATA / 'data.yaml').is_file() or dataset_counts(DATA) != EXPECTED:
    stage = WORK / 'descarga_temporal'
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir()
    if ZIP.is_file():
        with ZipFile(ZIP) as archive:
            for member in archive.infolist():
                if not (stage / member.filename).resolve().is_relative_to(stage.resolve()):
                    raise ValueError('El ZIP contiene una ruta que sale de la carpeta del dataset.')
            archive.extractall(stage)
        candidates = list(stage.rglob('data.yaml'))
        if len(candidates) != 1:
            raise ValueError('El ZIP debe contener un único data.yaml de Roboflow.')
        export = candidates[0].parent
    else:
        try:
            key = userdata.get('ROBOFLOW_API_KEY')
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            key = None
        if not key:
            raise RuntimeError('Habilita el secreto ROBOFLOW_API_KEY o guarda el ZIP oficial en Drive.')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow>=1.2,<2'])
        from roboflow import Roboflow

        export = stage / 'exportacion'
        try:
            project = Roboflow(api_key=key).workspace('remas-workspace').project('intersection-traffic-detection-sf6fy')
            project.version(1).download('yolov8', location=str(export))
        except Exception:
            raise RuntimeError('Roboflow no pudo exportar la v1. Revisa el acceso de la clave o usa el ZIP oficial.') from None
        finally:
            del key
    check_export(export)
    if DATA.exists():
        shutil.rmtree(DATA)
    if export == stage:
        stage.rename(DATA)
    else:
        export.rename(DATA)
        shutil.rmtree(stage)

print('Imágenes recibidas:', check_export(DATA))
if not ZIP.is_file():
    archive_path = Path(shutil.make_archive(str(WORK / 'dataset_semaforoia_yolov8'), 'zip', DATA))
    copy_atomic(archive_path, ZIP)
    archive_path.unlink()

drive_marker = DRIVE_DATA / '.copia_completa'
if not drive_marker.is_file():
    shutil.copytree(DATA, DRIVE_DATA, dirs_exist_ok=True)
    check_export(DRIVE_DATA)
    drive_marker.touch()
print('Dataset en Drive:', DRIVE_DATA)
print(f'ZIP: {ZIP.name} | {ZIP.stat().st_size / 1024**2:.1f} MiB')

### 8. Auditoría y control de solapamientos

Revisamos la relación entre imágenes y etiquetas, el rango de las coordenadas, las clases y los archivos idénticos. Si se dispone de los identificadores de video o escena, `grupos_origen.csv` en Drive puede agrupar todos los cuadros relacionados. La protección frente a cuadros visualmente cercanos de una misma secuencia depende de conocer ese origen. Las imágenes del conjunto de prueba se mantienen reservadas para la evaluación final. Las coincidencias detectadas con prueba se excluyen del desarrollo y se registran.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

original = yaml.safe_load((DATA / 'data.yaml').read_text(encoding='utf-8'))
raw_names = original['names']
NAMES = dict(enumerate(raw_names)) if isinstance(raw_names, list) else {
    int(index): name for index, name in raw_names.items()}
if set(NAMES) != set(range(len(NAMES))) or original.get('nc', len(NAMES)) != len(NAMES):
    raise ValueError('La numeración de clases del YAML no es consecutiva.')

records = []
for split in EXPECTED:
    images = sorted(p for p in (DATA / split / 'images').iterdir()
                    if p.is_file() and p.suffix.lower() in IMAGE_TYPES)
    labels = {p.stem: p for p in (DATA / split / 'labels').glob('*.txt')}
    stems = {p.stem for p in images}
    if len(stems) != len(images) or labels.keys() - stems:
        raise ValueError(f'Hay nombres de imagen ambiguos o etiquetas sin imagen en {split}.')
    for image in images:
        label = labels.get(image.stem)
        class_ids = []
        for row in label.read_text(encoding='utf-8').splitlines() if label else ():
            if not row.strip():
                continue
            parts = row.split()
            if len(parts) != 5:
                raise ValueError(f'Cada etiqueta debe tener clase y cuatro coordenadas: {label.name}')
            cls = int(parts[0])
            x, y, width, height = map(float, parts[1:])
            if cls not in NAMES or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < width <= 1 and 0 < height <= 1):
                raise ValueError(f'Clase o coordenadas YOLO fuera de rango: {label.name}')
            class_ids.append(cls)
        records.append({'split': split, 'path': str(image.relative_to(DATA)),
                        'hash': hashlib.sha256(image.read_bytes()).hexdigest(),
                        'label_exists': label is not None, 'classes': class_ids})

table = pd.DataFrame(records)
origins_path = ROOT / 'grupos_origen.csv'
if origins_path.is_file():
    origins = pd.read_csv(origins_path, dtype=str, keep_default_na=False)
    if not {'imagen', 'grupo'}.issubset(origins) or origins['imagen'].duplicated().any():
        raise ValueError('grupos_origen.csv requiere columnas imagen y grupo, con rutas únicas.')
    if set(origins['imagen']) - set(table['path']):
        raise ValueError('grupos_origen.csv contiene rutas ajenas a la exportación.')
    source_groups = dict(zip(origins['imagen'], origins['grupo']))
else:
    source_groups = {}

# Unimos grupos de origen si dos rutas tienen el mismo contenido.
parents = {}

def representative(group):
    while group in parents:
        group = parents[group]
    return group

seen_hashes = {}
groups = []
for row in table.itertuples():
    source = source_groups.get(row.path, '').strip()
    group = f'origen:{source}' if source else f'sha256:{row.hash}'
    other = seen_hashes.setdefault(row.hash, group)
    first, second = representative(group), representative(other)
    if first != second:
        parents[max(first, second)] = min(first, second)
    groups.append(group)
table['group'] = [representative(group) for group in groups]

test = table[table['split'] == 'test'].copy()
valid = table[(table['split'] == 'valid') & ~table['group'].isin(test['group'])].copy()
train = table[(table['split'] == 'train') & ~table['group'].isin(test['group']) &
              ~table['group'].isin(valid['group'])].copy()
dev = table[(table['split'] != 'test') & ~table['group'].isin(test['group'])].copy().reset_index(drop=True)

used_paths = set(pd.concat([train, valid, test])['path'])
table['evaluacion_principal'] = table['path'].isin(used_paths)
table['grupo_origen_disponible'] = table['path'].map(source_groups).fillna('').str.strip().ne('')
table.to_csv(ROOT / 'auditoria' / 'inventario.csv', index=False)
summary = []
for split, rows in table.groupby('split'):
    boxes = Counter(cls for classes in rows['classes'] for cls in classes)
    summary.append({'particion': split, 'imagenes': len(rows), 'con_etiqueta': int(rows['label_exists'].sum()),
                    'sin_cajas': sum(not classes for classes in rows['classes']),
                    'grupos': rows['group'].nunique(),
                    'con_origen': int(rows['grupo_origen_disponible'].sum()),
                    'excluidas_por_solapamiento': int((~rows['evaluacion_principal']).sum()),
                    **{f'cajas_{name}': boxes[cls] for cls, name in NAMES.items()}})
audit = pd.DataFrame(summary)
audit.to_csv(ROOT / 'auditoria' / 'resumen.csv', index=False)
display(audit)

### 9. Preparación de particiones y validación cruzada

El modelo principal usa `train` y `valid` oficiales. Las 10 secuencias inferidas de los nombres están representadas en las tres particiones. La prueba oficial evalúa imágenes reservadas de esas secuencias. En validación cruzada retenemos secuencias completas del conjunto de desarrollo en cada pliegue y conservamos los grupos de origen y duplicados. Se exporta la cobertura por clase. El modelo principal usa 30 épocas y los tres modelos de validación cruzada usan 10 épocas cada uno, con la misma inicialización preentrenada.

In [ ]:
RUNS = ROOT / 'resultados' / RUN_ID
RUNS.mkdir(parents=True, exist_ok=True)
CONFIG = WORK / 'config' / RUN_ID
CONFIG.mkdir(parents=True, exist_ok=True)

def dataset_config(name, training, validation, testing=None):
    splits = {'train': training, 'val': validation}
    if testing is not None:
        splits['test'] = testing
    config = {'path': str(DATA), 'names': NAMES}
    for split, rows in splits.items():
        if rows.empty:
            raise ValueError(f'La partición {name}/{split} quedó vacía.')
        listing = CONFIG / f'{name}_{split}.txt'
        listing.write_text('\n'.join(str(DATA / path) for path in rows['path']) + '\n', encoding='utf-8')
        config[split] = str(listing)
    target = CONFIG / f'{name}.yaml'
    target.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    return target

MAIN_YAML = dataset_config('principal', train, valid, test)

# Los nombres de Roboflow conservan cámara, fecha y hora de cada grabación.
sequence_pattern = r'/([^/]+)_\d+_(?:jpg|jpeg|png)\.rf\.[a-fA-F0-9]+\.[^.]+$'
recordings = table['path'].str.extract(sequence_pattern, expand=False)
sequence_audit = pd.DataFrame({'particion': table['split'], 'secuencia': recordings})
sequence_counts = pd.crosstab(sequence_audit['secuencia'], sequence_audit['particion'])
sequence_counts.to_csv(RUNS / 'secuencias_particiones_originales.csv')
display(sequence_counts)
if recordings.isna().any():
    print(f'{recordings.isna().sum()} imágenes carecen del patrón de secuencia de Roboflow.')

# Un grupo manual o un duplicado que vincula dos secuencias también debe quedar unido.
cv_parents = {}
def cv_root(group):
    while group in cv_parents:
        group = cv_parents[group]
    return group

for data_group, sequence in zip(table['group'], recordings):
    if pd.notna(sequence):
        first, second = cv_root(data_group), cv_root(f'secuencia:{sequence}')
        if first != second:
            cv_parents[max(first, second)] = min(first, second)
dev['cv_group'] = dev['group'].map(cv_root)
if dev['cv_group'].nunique() < FOLDS:
    raise ValueError('Hay menos grupos de secuencia que pliegues solicitados.')

frequency = Counter(cls for classes in dev['classes'] for cls in set(classes))
strata = dev['classes'].map(lambda ids: min(ids, key=lambda cls: (frequency[cls], cls)) if ids else -1)
folds = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_configs, assignments, coverage = [], [], []
for fold, (fit_idx, val_idx) in enumerate(folds.split(dev, strata, dev['cv_group']), 1):
    fit, validation = dev.iloc[fit_idx], dev.iloc[val_idx]
    assert not set(fit['cv_group']).intersection(validation['cv_group'])
    assert not set(validation['group']).intersection(test['group'])
    name = f'pliegue_secuencia_{fold:02}'
    fold_configs.append((name, dataset_config(name, fit, validation)))
    assignments.append(validation[['path', 'cv_group']].assign(pliegue=fold))
    for split, rows in [('train', fit), ('valid', validation)]:
        boxes = Counter(cls for classes in rows['classes'] for cls in classes)
        coverage.append({'pliegue': fold, 'particion': split, 'imagenes': len(rows),
                         'grupos': rows['cv_group'].nunique(),
                         **{f'cajas_{name}': boxes[cls] for cls, name in NAMES.items()}})

membership = pd.concat(assignments, ignore_index=True)
assert len(membership) == len(dev) and membership['path'].is_unique
membership.to_csv(RUNS / 'particiones_cv.csv', index=False)
coverage_report = pd.DataFrame(coverage)
coverage_report.to_csv(RUNS / 'cobertura_cv.csv', index=False)
shutil.copytree(CONFIG, RUNS / 'config', dirs_exist_ok=True)
display(coverage_report)

### 10. Fine-tuning con YOLOv8n

Cada época guarda `last.pt` y una copia con el estado del optimizador para reanudar si Colab se desconecta. `best.pt` se selecciona usando el conjunto de validación. Las curvas muestran pérdidas, precisión, recall y mAP. Los checkpoints periódicos se conservan cada cinco épocas. Para repetir un experimento con otra configuración, cambia `RUN_ID`.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image

def run_training(name, config, epochs):
    if not torch.cuda.is_available():
        raise RuntimeError('Colab aún no tiene una GPU asignada. Selecciona GPU T4 y vuelve a ejecutar.')

    folder = RUNS / name
    weights = folder / 'weights'
    best, resume = weights / 'best.pt', weights / 'resume.pt'
    completed = folder / 'completado.json'
    fingerprint = hashlib.sha256(config.read_bytes())
    data_paths = yaml.safe_load(config.read_text(encoding='utf-8'))
    image_hashes = table.set_index('path')['hash'].to_dict()
    for split in ('train', 'val', 'test'):
        if split in data_paths:
            for line in Path(data_paths[split]).read_text(encoding='utf-8').splitlines():
                relative = Path(line).relative_to(DATA).as_posix()
                fingerprint.update(f'{relative} {image_hashes[relative]}\n'.encode())
    spec = dict(semilla=SEED, epocas=epochs, imgsz=IMGSZ, batch=BATCH, base='yolov8n.pt',
                config_sha256=fingerprint.hexdigest(), ultralytics=ultralytics.__version__)
    folder.mkdir(parents=True, exist_ok=True)
    manifest = folder / 'experimento.json'
    if manifest.exists() and json.loads(manifest.read_text(encoding='utf-8')) != spec:
        raise ValueError('La configuración cambió. Define otro RUN_ID y vuelve a ejecutar desde el bloque 9.')
    write_json(manifest, spec)
    if completed.is_file() and best.is_file():
        print(f'{name}: resultados existentes en {folder}')
        return best

    def save_resume(trainer):
        copy_atomic(Path(trainer.last), resume)

    if resume.is_file():
        # Si hubo un corte tras guardar la última época, el modelo ya realizó el ajuste.
        checkpoint = torch.load(resume, map_location='cpu', weights_only=False)
        finished_epochs = checkpoint['epoch'] + 1
        del checkpoint
        if finished_epochs >= epochs and best.is_file():
            write_json(completed, spec)
            return best
        model = YOLO(str(resume))
        model.add_callback('on_model_save', save_resume)
        model.train(resume=True, device=0)
    else:
        model = YOLO('yolov8n.pt')
        model.add_callback('on_model_save', save_resume)
        model.train(data=str(config), epochs=epochs, imgsz=IMGSZ, batch=BATCH, device=0,
                    seed=SEED, deterministic=True, workers=2, amp=True, patience=0,
                    project=str(RUNS), name=name, exist_ok=True, plots=True, save_period=5,
                    close_mosaic=min(10, max(0, epochs - 1)))
    if not best.is_file():
        raise FileNotFoundError(f'No se generó {best}.')
    write_json(completed, spec)
    return best

BEST = run_training('principal', MAIN_YAML, EPOCHS)
APP_MODEL = ROOT / 'modelos' / 'best.pt'
copy_atomic(BEST, APP_MODEL)
history = BEST.parent.parent / 'results.csv'
if history.is_file():
    display(pd.read_csv(history).tail())
training_plot = history.with_name('results.png')
if training_plot.is_file():
    display(Image(filename=str(training_plot), width=900))
print('Modelo para Streamlit:', APP_MODEL)

### 11. Validación cruzada y matrices de confusión

Evaluamos cada pliegue con los pesos de su última época para comparar un presupuesto fijo. La media y la desviación estándar describen cuánto variaron las métricas entre pliegues. Las matrices de confusión muestran clases confundidas, falsos positivos y falsos negativos según el criterio de asociación de Ultralytics.

In [ ]:
def evaluate(weights, config, split, name):
    saved = RUNS / 'evaluacion' / name
    if all((saved / filename).is_file() for filename in
           ('metricas.json', 'metricas_por_clase.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png')):
        print(f'{name}: evaluación existente en {saved}')
        return json.loads((saved / 'metricas.json').read_text(encoding='utf-8'))
    result = YOLO(str(weights)).val(data=str(config), split=split, device=0,
                                    imgsz=IMGSZ, batch=8, plots=True,
                                    project=str(RUNS / 'evaluacion'), name=name, exist_ok=True)
    folder = Path(result.save_dir)
    metrics = {'precision': float(result.box.mp), 'recall': float(result.box.mr),
               'mAP50': float(result.box.map50), 'mAP50_95': float(result.box.map)}
    write_json(folder / 'metricas.json', metrics)
    per_class = []
    for index, cls in enumerate(result.box.ap_class_index):
        precision, recall, ap50, ap = result.box.class_result(index)
        per_class.append({'clase': NAMES[int(cls)], 'precision': float(precision),
                          'recall': float(recall), 'mAP50': float(ap50), 'mAP50_95': float(ap)})
    pd.DataFrame(per_class).to_csv(folder / 'metricas_por_clase.csv', index=False)
    return metrics

cv_rows = []
for name, config in fold_configs:
    fold_best = run_training(name, config, CV_EPOCHS)
    last = fold_best.with_name('last.pt')
    cv_rows.append({'pliegue': name, **evaluate(last, config, 'val', name)})
    pd.DataFrame(cv_rows).to_csv(RUNS / 'metricas_cv.csv', index=False)

cv_table = pd.DataFrame(cv_rows).set_index('pliegue')
cv_summary = cv_table.agg(['mean', 'std']).rename(index={'mean': 'media', 'std': 'desviacion_estandar'})
cv_summary.to_csv(RUNS / 'resumen_cv.csv')
display(cv_table, cv_summary)

### 12. Prueba final del modelo principal

Aplicamos el `best.pt` elegido en validación al conjunto de prueba reservado. Reportamos precisión, recall, mAP a IoU 0,50 y mAP promediado de IoU 0,50 a 0,95. Se guardan las métricas por clase, las curvas y las matrices de confusión en Drive.

In [ ]:
test_metrics = evaluate(BEST, MAIN_YAML, 'test', 'prueba_final')
display(pd.DataFrame([test_metrics], index=['prueba']))
test_folder = RUNS / 'evaluacion' / 'prueba_final'
for filename in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
    chart = test_folder / filename
    if chart.is_file():
        display(Image(filename=str(chart), width=650))

### 13. Descargar best.pt

Ejecuta esta celda para guardar el modelo en tu computadora y ubícalo en `models/best.pt` dentro del proyecto Streamlit.

In [ ]:
download_button(ROOT / 'modelos' / 'best.pt', 'Descargar best.pt')

### 14. Descargar ZIP del dataset

El ZIP incluye `data.yaml`, imágenes y etiquetas de las tres particiones. También puede descargarse directamente desde Drive.

In [ ]:
download_button(ZIP, 'Descargar ZIP del dataset')

### Cierre de la evaluación del experimento v1_semilla42

**23 de noviembre de 2026.** Se conservaron los tres entrenamientos por secuencia, de 10 épocas cada uno. Los pliegues 1 y 2 se evaluaron en T4. La evaluación del tercer pliegue y la prueba oficial se completaron por inferencia en CPU con los pesos y las listas originales guardados en Drive.

- **Prueba oficial (671 imágenes).** Precisión 55,35 %, sensibilidad 57,52 %, mAP a IoU 0,50 de 61,61 % y mAP entre IoU 0,50 y 0,95 de 50,22 %.
- **Validación por secuencia (3 pliegues).** mAP medio entre IoU 0,50 y 0,95 de 30,31 %, con desviación estándar de 1,74 puntos porcentuales.

[Resultados y resumen de los pliegues](https://drive.google.com/drive/folders/1Nkbtqtoq7bVMVcbGc36zG1mtvddzO-6u) · [Métricas y matrices de la prueba oficial](https://drive.google.com/drive/folders/1Vt-T4Zyfx4-QO9VDq1uUxM-L65xRuuxL).